In [30]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']   #显示中文
plt.rcParams['axes.unicode_minus']=False       #显示负号
plt.rcParams['font.family'] = 'Times New Roman'

In [31]:
dif = pd.read_csv('DIF.csv')

In [32]:
select_cols_x = ['DIF','SIZE',	'LEV',	'TOP'	,'GROWTH',	'CFO'	,'AGE'	,'BSIZE'] # 'Innovation','ICL','FC',
select_cols_y = ['TFPLP',]
select_cols = select_cols_x + select_cols_y 

In [33]:
groups = dict(list(dif.groupby('证券简称')))

In [34]:
groups.keys()

dict_keys(['世纪瑞尔', '东山精密', '东港股份', '中南建设', '中国神华', '中泰化学', '中航电测', '二六三', '云铝股份', '京新药业', '信维通信', '信邦制药', '凤竹纺织', '北斗星通', '华天科技', '华微电子', '华测检测', '南山铝业', '四川双马', '四川成渝', '四环生物', '国中水务', '国电南自', '塔牌集团', '大禹节水', '天源迪科', '威海广泰', '宁夏建材', '安科生物', '宏达股份', '宝钛股份', '岳阳林纸', '川润股份', '巨力索具', '广电网络', '建新股份', '开创国际', '恒星科技', '数字政通', '新疆众和', '昌红科技', '杭萧钢构', '桂东电力', '民丰特纸', '永安药业', '汉王科技', '汤臣倍健', '沃华医药', '沃尔核材', '沃森生物', '海油工程', '漫步者', '爱尔眼科', '现代投资', '皖维高新', '皖能电力', '益生股份', '神州泰岳', '立讯精密', '羚锐制药', '联美控股', '航天机电', '英飞拓', '莱宝高科', '誉衡药业', '达华智能', '通化东宝', '金晶科技', '金鹰股份', '银之杰', '银河磁体', '长盈精密', '陕鼓动力', '顺网科技', '驰宏锌锗', '高德红外', '高新兴'])

In [35]:
all_df_list = []
for k in groups.keys():
    df = groups[k].loc[:,select_cols+['year']]
    df = df.sort_values('year')
    df = df.loc[:,select_cols    ]
    groups[k] = df
    all_df_list.append(df)

In [36]:
test_df = groups['世纪瑞尔']

In [37]:
# test_df.plot(subplots=True, figsize=(15, 10))
# plt.savefig('test_df.png',dpi=300)

In [38]:
LOOK_BACK = 2
train_company = np.random.choice(a=list(groups.keys()),size=int(len(groups.keys())*0.7))
test_company = [i for i in groups.keys() if i not in train_company]

In [39]:
from sklearn.preprocessing import MinMaxScaler

def create_sequences(input_data, seq_length):
    Xs, ys,masks = [], [], []
    for i in range(len(input_data) - seq_length):
        seq_x = input_data[i:(i + seq_length)]
        seq_y = input_data[i + seq_length][-1]
        Xs.append(seq_x)
        ys.append(seq_y)
    return np.array(Xs), np.array(ys)


In [40]:
X_train, y_train = [],[]
scaler = MinMaxScaler(feature_range=(-1, 1))
for c_n in train_company:
    train_df = groups[c_n]
    # Train
    current_data = pd.concat([pd.concat([train_df.iloc[[0],:] for i in range(LOOK_BACK)]),train_df],axis=0)
    x,y = create_sequences(scaler.fit_transform(current_data.values),LOOK_BACK)
    X_train.append(x)
    y_train.append(y)

X_train = np.concatenate(X_train,axis=0)
y_train = np.concatenate(y_train,axis=0)
X_train = np.expand_dims(X_train,axis=-1)
X_train = np.nan_to_num(X_train,0)
y_train = np.nan_to_num(y_train,0)

X_test = {}
for c_n in test_company:
    scaler = MinMaxScaler(feature_range=(-1, 1))
    test_df = groups[c_n]
    # Train
    current_data = pd.concat([pd.concat([test_df.iloc[[0],:] for i in range(LOOK_BACK)]),test_df],axis=0)
    x,y = create_sequences(scaler.fit_transform(current_data.values),LOOK_BACK)
    X_test[c_n] = {'x':x,'y':y,'scaler_max':scaler.data_max_,'scaler_min':scaler.data_min_}


In [41]:
adj = np.ones(shape=(9,9)).astype(int)

In [42]:
np.savez('./data.npz',X_train=X_train,y_train=y_train,X_test=X_test,adj=adj)

In [43]:
current_data

,DIF,SIZE,LEV,TOP,GROWTH,CFO,AGE,BSIZE,TFPLP
468,69.48000,20.757453,0.146348,49.78,0.075706,0.003250,1.945910,1.945910,6.606781
468,69.48000,20.757453,0.146348,49.78,0.075706,0.003250,1.945910,1.945910,6.606781
468,69.48000,20.757453,0.146348,49.78,0.075706,0.003250,1.945910,1.945910,6.606781
469,127.06000,20.738439,0.180962,49.78,0.541015,-0.025618,1.945910,1.945910,6.986225
470,184.78000,20.923604,0.248925,48.10,0.776430,-0.112919,1.945910,1.945910,7.571042
471,201.53000,21.235422,0.390728,42.14,0.373489,-0.084169,2.079442,1.945910,7.910162
472,240.95000,22.180812,0.202109,35.78,0.465545,-0.027676,2.197225,1.945910,8.234011
473,247.99617,22.298158,0.217624,35.65,0.210138,0.057895,2.302585,1.945910,8.336745
474,296.16794,22.742406,0.315676,31.58,0.710696,0.014954,2.397895,2.197225,8.712162
475,331.91759,22.845550,0.319907,30.52,0.592669,-0.018565,2.484907,2.197225,9.096401
